# Export images for re-annotation (MOT 1.0 format)

Exports selected bags into per-bag folders that mirror the existing MOT sequence structure:

```
for_annotation/
└── bag1/
│   ├── frames/   ← images
│   └── gt/
│       ├── gt.txt      ← current annotations in MOT 1.0 format
│       └── labels.txt  ← class names (1-indexed)
└── bag9/
    ├── frames/
    └── gt/
        ├── gt.txt
        └── labels.txt
```

**gt.txt column order:** `frame_id, track_id, x, y, w, h, 1, class_id, 1.0`
- Coordinates are absolute pixels (top-left origin).
- `track_id` is assigned per-object per-frame (no cross-frame tracking in source data).
- `class_id` is 1-indexed: `1=fish`, `2=net`.

In [1]:
from pathlib import Path
import shutil

NOTEBOOK_DIR = Path.cwd()
SPLIT_IMAGES = NOTEBOOK_DIR / "processed" / "images"
SPLIT_LABELS = NOTEBOOK_DIR / "processed" / "labels"

# ---- config ----
EXPORT_BAGS = ["bag1", "bag9"]
OUT_DIR     = NOTEBOOK_DIR / "for_annotation"
IMAGE_EXTS  = {".jpg", ".jpeg", ".png", ".bmp"}
IMG_W, IMG_H = 1200, 700

# MOT class names (1-indexed, line N = class_id N)
CLASS_NAMES = ["fish", "net"]

print(f"Output : {OUT_DIR}")
print(f"Bags   : {EXPORT_BAGS}")

Output : /cluster/home/henrban/aquaculture-perception/data-processing/sonar/net_fish_sonar_improved/for_annotation
Bags   : ['bag1', 'bag9']


In [2]:
def yolo_to_mot_row(frame_id, track_id, yolo_line, img_w, img_h):
    """Convert one YOLO annotation line to a MOT 1.0 gt.txt row string."""
    parts      = yolo_line.strip().split()
    yolo_cls   = int(parts[0])
    xc, yc, wn, hn = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
    x = (xc - wn / 2) * img_w
    y = (yc - hn / 2) * img_h
    w = wn * img_w
    h = hn * img_h
    mot_cls = yolo_cls + 1          # YOLO 0-indexed → MOT 1-indexed
    return f"{frame_id},{track_id},{x:.2f},{y:.2f},{w:.2f},{h:.2f},1,{mot_cls},1.0"


for bag in EXPORT_BAGS:
    frames_out = OUT_DIR / bag / "frames"
    gt_out     = OUT_DIR / bag / "gt"
    if (OUT_DIR / bag).exists():
        shutil.rmtree(OUT_DIR / bag)
    frames_out.mkdir(parents=True)
    gt_out.mkdir(parents=True)

    # collect images for this bag across all splits, sorted chronologically
    bag_images = sorted(
        p for p in SPLIT_IMAGES.rglob("*")
        if p.suffix.lower() in IMAGE_EXTS and p.stem.startswith(f"{bag}_")
    )

    gt_rows = []
    for frame_id, img_path in enumerate(bag_images, start=1):
        shutil.copy2(img_path, frames_out / img_path.name)

        split    = img_path.parent.name           # "train" / "val" / "test"
        lbl_path = SPLIT_LABELS / split / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            continue
        lines = [l for l in lbl_path.read_text(encoding="utf-8").splitlines() if l.strip()]
        for track_id, line in enumerate(lines, start=1):
            gt_rows.append(yolo_to_mot_row(frame_id, track_id, line, IMG_W, IMG_H))

    (gt_out / "gt.txt").write_text("\n".join(gt_rows), encoding="utf-8")
    (gt_out / "labels.txt").write_text("\n".join(CLASS_NAMES), encoding="utf-8")

    print(f"  {bag}: {len(bag_images)} frames, {len(gt_rows)} annotations")

  bag1: 200 frames, 269 annotations
  bag9: 200 frames, 275 annotations


Annotations where updated/imporved and uploaded again. Used CVAT for labeling, where we can export on the yolo-format. 